In [ ]:
# @title 1. Setup Environment
import os

# Check GPU
!nvidia-smi

# Clone the Hugging Face Space directly to ensure we get app.py
# (We rename the folder to 'echo_tts_space' to avoid confusion)
if not os.path.exists("echo_tts_space"):
    print("Cloning Hugging Face Space...")
    !git clone https://huggingface.co/spaces/jordand/echo-tts-preview echo_tts_space

# Install System Dependencies
!sudo apt-get install -y ffmpeg

# Install Python Dependencies
# We enter the directory to install requirements, then go back
%cd echo_tts_space
!pip install -r requirements.txt
!pip install --pre torchcodec --index-url https://download.pytorch.org/whl/nightly/cu121
%cd ..

In [ ]:
# @title 2. Hugging Face Login (Optional)
import os
from google.colab import userdata

# Paste token here or leave blank if using Colab Secrets
HF_TOKEN_INPUT = "" # @param {type:"string"}

if not HF_TOKEN_INPUT:
    try:
        HF_TOKEN_INPUT = userdata.get('HF_TOKEN')
    except:
        pass

if HF_TOKEN_INPUT:
    os.environ["HF_TOKEN"] = HF_TOKEN_INPUT
    from huggingface_hub import login
    login(token=HF_TOKEN_INPUT)
    print("Logged in successfully.")
else:
    print("Proceeding without login.")

In [ ]:
# @title 3. Run Echo-TTS (Fixed Repo ID)
import sys
import os
from types import ModuleType
from pathlib import Path

# --- FIX 1: Set Directory explicitly ---
repo_dir = "/content/echo_tts_space"

if not os.path.exists(repo_dir):
    print(f"Error: {repo_dir} does not exist. Please run Cell 1 again.")
else:
    # Change directory so app.py finds local assets
    os.chdir(repo_dir)
    # Add to system path
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)
    print(f"Working directory set to: {os.getcwd()}")

# --- FIX 2: Patch inference.py to use the correct Repo ID ---
# The original code points to 'jordand/echo-tts' which causes the 404 error.
# We replace it with 'jordand/echo-tts-base' where the file actually exists.
inference_file = Path("inference.py")
if inference_file.exists():
    print("Patching inference.py with correct repository URL...")
    content = inference_file.read_text()
    # Replace the default argument
    new_content = content.replace("repo_id: str = 'jordand/echo-tts'", "repo_id: str = 'jordand/echo-tts-base'")
    inference_file.write_text(new_content)
else:
    print("Warning: inference.py not found, skipping patch (this might cause errors).")

# --- FIX 3: Mock the 'spaces' library ---
# Colab doesn't have ZeroGPU, so we fake the library.
if "spaces" not in sys.modules:
    m = ModuleType("spaces")
    def GPU(func):
        return func
    m.GPU = GPU
    sys.modules["spaces"] = m

# --- Setup Output Directories ---
TEMP_AUDIO_DIR = Path('./temp_gradio_audio')
TEMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# --- Import and Launch ---
print("Importing app... Models will download now (this may take 1-2 minutes)...")

try:
    from app import demo
    print("App loaded successfully. Launching...")
    demo.queue(max_size=20).launch(share=True, debug=True)
except ImportError as e:
    print("----------------------------------------------------------------")
    print(f"Failed to import app: {e}")
    print("Please check that the 'echo_tts_space' folder exists.")
    print("----------------------------------------------------------------")